In [ ]:
import sys
import subprocess
import os
import glob
from PyQt5.QtCore import Qt
from PyQt5.QtWidgets import QStackedWidget, QMainWindow, QApplication, QWidget, QVBoxLayout, QFormLayout, QLineEdit, QPushButton, QHBoxLayout, QRadioButton, QCheckBox, QButtonGroup, QComboBox, QLabel


class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("参数输入界面")
        self.setGeometry(100, 100, 600, 500)
        # 英文变量名到中文标签的映射
        self.param_labels = {
            "geometry": "几何结构",
            "colling_type": "冷却类型",
            "material": "材料参数",
            "heat_flux": "负载",
            "frequency": "频率(Hz)",
            "temperature": "温度(C)",
            "conv_center": f"中心对流(W/m2·C)",
            "conv_side": f"边缘对流(W/m2·C)",
            "l": "长度 l(m)",
            "b": "宽度 b(m)",
            "t": "厚度 t(m)",
            "OFHC_L_mid": "中部铜管长度(m)",
            "OFHC_L_side": "侧部铜管长度(m)",
            "GAP_CU": "铜管间距(m)",
            "dw_length": "窗口长度(m)",
            "kong_height": "孔高度(m)",
            "kong_length": "孔长度(m)",
            "notch_depth": "凹槽深度(m)",
            "kaicao": "开槽",
            "cao_optics": "开槽位置(m)",
            "cao_kuan": "开槽宽度(m)",
            "cao_height": "开槽高度(m)",
            "num_cores": "核心数量",
            "optics_face(mm)": "几何精度(mm)",
            "ns_inga(mm)": "ns InGa(mm)",
            "ns_mirror(mm)": "ns 镜面(mm)",
            "ns_cu(mm)": "ns 铜(mm)"
        }


        # 创建 QStackedWidget
        self.stacked_widget = QStackedWidget()
        self.setCentralWidget(self.stacked_widget)

        # 创建两个页面
        self.page1 = QWidget()
        self.page2 = QWidget()

        # 初始化页面内容
        self.init_page1()
        self.init_page2()

        # 添加页面到堆叠控件了，
        self.stacked_widget.addWidget(self.page1)
        self.stacked_widget.addWidget(self.page2)
        

    def init_page1(self):
        # 定义参数名称和默认值
        self.param_names = [
            "geometry", "colling_type","material", 
            "l", "b", "t", 
            "OFHC_L_mid", "OFHC_L_side", "GAP_CU", 
            "dw_length", "kong_height", "kong_length", "notch_depth", 
            "kaicao", "cao_optics", "cao_kuan", "cao_height", 
            "l_optics", "b_optics",
            "num_cores"
        ]
        
        self.default_values = [
            ("M1"), ("INGA 3"), ("SYS"),
            ("0.85", "1.1", "0.01"), ("0.05", "0.071", "0.001"), ("0.06", "0.061", "0.001"),
            ("0.10", "0.201", "0.005"), ("0.10", "0.201", "0.005"), ("0.005", "0.06", "0.005"),
            ("0.03", "0.04", "0.01"), ("0.013", "0.014", "0.001"), ("0.025", "0.026", "0.001"), ("0.000", "0.0081", "0.001"), 
            ("yes"), ("0.031"), ("0.007"), ("0.007"),
            ("0.85"), ("0.046"),
            ("5")
        ]
        
        layout = QVBoxLayout()

        # 创建表单布局 
        form_layout = QFormLayout()
        # 创建输入框并添加到布局
        self.inputs = {}
        self.checkboxes = {}  # 存储复选框
        self.radio_buttons = {}  # 存储 "yes"/"no" 按钮
        self.radio_buttons1 = {}  # 存储 "yes"/"no" 按钮
        self.button_groups = {}  # 存储按钮组
        
        # "geometry" 和 "material" 参数只有一个输入框
        for i, name in enumerate(self.param_names):

            if name in ["material", "num_cores"]:
                # 只添加一个输入框，不包含多值复选框
                input_widget = QLineEdit(self.default_values[i])
                self.inputs[name] = input_widget
                form_layout.addRow(self.param_labels.get(name, name), input_widget)  # 直接显示输入框，不添加复选框
                
            elif name == "colling_type":
                # 创建下拉选择框
                combo_box = QComboBox()
                combo_box.addItem("INGA 1")
                combo_box.addItem("INGA 3")
                combo_box.addItem("IN 1")
                combo_box.addItem("IN 3")
                # 设置默认值
                combo_box.setCurrentText(self.default_values[i])
                self.inputs[name] = combo_box
                form_layout.addRow(self.param_labels.get(name, name), combo_box)  # 添加到布局中
                
            elif name == "geometry":
                # 创建下拉选择框
                combo_box = QComboBox()
                combo_box.addItem("M1")
                combo_box.addItem("M2")
                combo_box.addItem("M3")
                combo_box.addItem("M4")
                combo_box.addItem("M4a")
                combo_box.addItem("M4b")
                combo_box.addItem("G1a")
                combo_box.addItem("G1b")
                combo_box.addItem("M5")
                combo_box.addItem("M6")
                combo_box.addItem("KB-plane")
                combo_box.addItem("KB1-h")
                combo_box.addItem("KB2-h")
                combo_box.addItem("KB3-h")
                combo_box.addItem("KB1-v")
                combo_box.addItem("KB2-v")
                combo_box.addItem("KB3-v")
                # 设置默认值
                combo_box.setCurrentText(self.default_values[i])
                self.inputs[name] = combo_box
                form_layout.addRow(self.param_labels.get(name, name), combo_box)  # 添加到布局中

            elif name in ["kaicao", "cao_optics", "cao_kuan", "cao_height"]:
                if name == "kaicao":
                    h_layout = QHBoxLayout()

                    # 创建下拉选择框
                    combo_box = QComboBox()
                    combo_box.addItem("yes")
                    combo_box.addItem("no")
                    # 设置默认值
                    combo_box.setCurrentText(self.default_values[i])
                    self.inputs[name] = combo_box
                    #form_layout.addRow(self.param_labels.get(name, name), combo_box)  # 添加到布局中
                    h_layout.addWidget(QLabel(self.param_labels.get(name, name)))
                    h_layout.addWidget(combo_box)



                    input_widget = QLineEdit(self.default_values[i+1])
                    self.inputs["cao_optics"] = input_widget
                    h_layout.addWidget(QLabel("开槽位置(m)"))
                    h_layout.addWidget(input_widget)
                    
                    input_widget = QLineEdit(self.default_values[i+2])
                    self.inputs["cao_kuan"] = input_widget
                    h_layout.addWidget(QLabel("开槽宽度(m)"))
                    h_layout.addWidget(input_widget)
                    
                    input_widget = QLineEdit(self.default_values[i+3])
                    self.inputs["cao_height"] = input_widget
                    h_layout.addWidget(QLabel("开槽深度(m)"))
                    h_layout.addWidget(input_widget)

                    form_layout.addRow(h_layout)

            elif name in ["l_optics", "b_optics"]:
                if name == "l_optics":
                    h_layout = QHBoxLayout()

                    input_widget = QLineEdit(self.default_values[i])
                    self.inputs["l_optics"] = input_widget
                    h_layout.addWidget(QLabel("光学面长度(m)"))
                    h_layout.addWidget(input_widget)
                    
                    input_widget = QLineEdit(self.default_values[i+1])
                    self.inputs["b_optics"] = input_widget
                    h_layout.addWidget(QLabel("光学面宽度(m)"))
                    h_layout.addWidget(input_widget)


                    form_layout.addRow(h_layout)

            else:
                start_input = QLineEdit(self.default_values[i][0])
                end_input = QLineEdit(self.default_values[i][1])
                step_input = QLineEdit(self.default_values[i][2])
                # 创建复选框，文本修改为“多值”
                checkbox = QCheckBox("单值")
                checkbox.setChecked(False)  # 默认为未选中
                # 连接复选框的状态变化信号
                checkbox.stateChanged.connect(lambda state, step_input=step_input, end_input=end_input, i=i: self.on_checkbox_state_changed(state, step_input, end_input, i))
                self.inputs[name] = (start_input, end_input, step_input, checkbox)
                self.checkboxes[name] = checkbox

                # 将输入框和复选框放到同一行
                h_layout = QHBoxLayout()
                h_layout.addWidget(start_input)
                h_layout.addWidget(end_input)
                h_layout.addWidget(step_input)
                h_layout.addWidget(checkbox)
                
                form_layout.addRow(self.param_labels.get(name, name), h_layout)  # 使用 `name` 显示标签

        # 添加标签
        label = QLabel("构建几何结构")

        # 保存按钮
        self.save_button = QPushButton("保存")
        self.save_button.clicked.connect(self.save_to_txt1)

        # 运行按钮
        self.run_button = QPushButton("运行 (单核)")
        self.run_button.clicked.connect(self.save_to_txt1)
        self.run_button.clicked.connect(self.run_python_file1)

        # 运行按钮
        self.run_button2 = QPushButton("运行 (多核)")
        self.run_button2.clicked.connect(self.save_to_txt1)
        self.run_button2.clicked.connect(self.run_python_file_mpiexec1)
        
        # 删除文件按钮
        self.delete_button = QPushButton("清除所有")
        self.delete_button.clicked.connect(self.delete_all_result)

        # 退出按钮
        self.exit_button = QPushButton("退出")
        self.exit_button.clicked.connect(self.close)  # 点击时关闭窗口

        # 创建垂直布局
        layout = QVBoxLayout()
        layout.addWidget(label)
        layout.addLayout(form_layout)
        layout.addWidget(self.save_button)  # 添加保存按钮
        layout.addWidget(self.run_button)  # 添加运行按钮
        layout.addWidget(self.run_button2)  # 添加运行按钮
        layout.addWidget(self.delete_button)  # 添加删除按钮
        

        # 添加按钮
        button = QPushButton("切换热分析")
        button.clicked.connect(self.show_page2)
        layout.addWidget(button)

        layout.addWidget(self.exit_button)  # 添加退出按钮

        self.page1.setLayout(layout)

    def init_page2(self):
        self.param_names[3:3] = ["geometry1", "colling_type1","material1", "heat_flux", "frequency", "temperature", "conv_center", "conv_side"]
        self.param_names.extend(["optics_face(mm)", "ns_inga(mm)", "ns_mirror(mm)", "ns_cu(mm)", "num_cores1"])

        self.default_values[3:3] = [("M1"), ("INGA 3"), ("SYS"), ("M1 EEHG"), ("100k"), ("22", "23", "0.1"), ("5000", "5001", "500"), ("5000", "5001", "500")]
        self.default_values.extend([("2"), ("4"), ("4"), ("4"), ("5")])
        
        layout = QVBoxLayout()

        # 创建表单布局
        form_layout = QFormLayout()
        # 创建输入框并添加到布局
        # self.inputs = {}
        # self.checkboxes = {}  # 存储复选框
        # self.radio_buttons = {}  # 存储 "yes"/"no" 按钮
        # self.radio_buttons1 = {}  # 存储 "yes"/"no" 按钮
        # self.button_groups = {}  # 存储按钮组
        
        # "geometry" 和 "material" 参数只有一个输入框
        for i, name in enumerate(self.param_names):
            if name in [
                "geometry", "colling_type","material", 
                "l", "b", "t", "OFHC_L_mid", "OFHC_L_side", "GAP_CU", "dw_length", "kong_height", "kong_length", "notch_depth", "l_optics", "b_optics",
                "kaicao", "cao_optics", "cao_kuan", "cao_height", "num_cores"]: 
                continue
            
            elif name in ["material1", "frequency","num_cores1"]:
                # 只添加一个输入框，不包含多值复选框
                input_widget = QLineEdit(self.default_values[i])
                self.inputs[name] = input_widget
                form_layout.addRow(self.param_labels.get(name, name), input_widget)  # 直接显示输入框，不添加复选框
                
            elif name == "colling_type1":
                # 创建下拉选择框
                combo_box = QComboBox()
                combo_box.addItem("INGA 1")
                combo_box.addItem("INGA 3")
                combo_box.addItem("IN 1")
                combo_box.addItem("IN 3")
                # 设置默认值
                combo_box.setCurrentText(self.default_values[i])
                self.inputs[name] = combo_box
                form_layout.addRow(self.param_labels.get(name, name), combo_box)  # 添加到布局中
                
            elif name == "geometry1":
                # 创建下拉选择框
                combo_box = QComboBox()
                combo_box.addItem("M1")
                combo_box.addItem("M2")
                combo_box.addItem("M3")
                combo_box.addItem("M4")
                combo_box.addItem("M4a")
                combo_box.addItem("M4b")
                combo_box.addItem("G1a")
                combo_box.addItem("G1b")
                combo_box.addItem("M5")
                combo_box.addItem("M6")
                combo_box.addItem("KB-plane")
                combo_box.addItem("KB1-h")
                combo_box.addItem("KB2-h")
                combo_box.addItem("KB3-h")
                combo_box.addItem("KB1-v")
                combo_box.addItem("KB2-v")
                combo_box.addItem("KB3-v")
                # 设置默认值
                combo_box.setCurrentText(self.default_values[i])
                self.inputs[name] = combo_box
                form_layout.addRow(self.param_labels.get(name, name), combo_box)  # 添加到布局中
                
            elif name == "heat_flux":
                # 创建下拉选择框
                combo_box = QComboBox()
                combo_box.addItem("EEHG")
                combo_box.addItem("EEHG 3")
                combo_box.addItem("EEHG taper 3")
                combo_box.addItem("SASE")
                combo_box.addItem("SASE 3")
                combo_box.addItem("SASE taper 3")
                # 设置默认值
                combo_box.setCurrentText(self.default_values[i])
                self.inputs[name] = combo_box
                form_layout.addRow(self.param_labels.get(name, name), combo_box)  # 添加到布局中
                
            elif name in ["optics_face(mm)", "ns_inga(mm)", "ns_mirror(mm)", "ns_cu(mm)"]:
                if name == "optics_face(mm)":
                    h_layout = QHBoxLayout()
                    input_widget = QLineEdit(self.default_values[i])
                    self.inputs[name] = input_widget
                    h_layout.addWidget(QLabel("光学面网格(mm)"))
                    h_layout.addWidget(input_widget)
                    
                    input_widget = QLineEdit(self.default_values[i+1])
                    self.inputs["ns_inga(mm)"] = input_widget
                    h_layout.addWidget(QLabel("铟镓网格(mm)"))
                    h_layout.addWidget(input_widget)
                    form_layout.addRow(h_layout)
                    
                    input_widget = QLineEdit(self.default_values[i+2])
                    self.inputs["ns_mirror(mm)"] = input_widget
                    h_layout.addWidget(QLabel("镜体网格(mm)"))
                    h_layout.addWidget(input_widget)
                    
                    input_widget = QLineEdit(self.default_values[i+3])
                    self.inputs["ns_cu(mm)"] = input_widget
                    h_layout.addWidget(QLabel("铜管(mm)"))
                    h_layout.addWidget(input_widget)
                    form_layout.addRow(h_layout)

            elif name in ["temperature", "conv_center", "conv_side"]:
                start_input = QLineEdit(self.default_values[i][0])
                end_input = QLineEdit(self.default_values[i][1])
                step_input = QLineEdit(self.default_values[i][2])
                # 创建复选框，文本修改为“多值”
                checkbox = QCheckBox("单值")
                checkbox.setChecked(False)  # 默认为未选中
                # 连接复选框的状态变化信号
                checkbox.stateChanged.connect(lambda state, step_input=step_input, end_input=end_input, i=i: self.on_checkbox_state_changed(state, step_input, end_input, i))
                self.inputs[name] = (start_input, end_input, step_input, checkbox)
                self.checkboxes[name] = checkbox

                # 将输入框和复选框放到同一行
                h_layout = QHBoxLayout()
                h_layout.addWidget(start_input)
                h_layout.addWidget(end_input)
                h_layout.addWidget(step_input)
                h_layout.addWidget(checkbox)
                
                form_layout.addRow(self.param_labels.get(name, name), h_layout)  # 使用 `name` 显示标签

        # 添加标签
        label = QLabel("热分析")

        # 保存按钮
        self.save_button = QPushButton("保存")
        self.save_button.clicked.connect(self.save_to_txt2)

        # 运行按钮
        self.run_button = QPushButton("运行 (单核)")
        self.run_button.clicked.connect(self.save_to_txt2)
        self.run_button.clicked.connect(self.run_python_file2)

        # 运行按钮
        self.run_button2 = QPushButton("运行 (多核)")
        self.run_button2.clicked.connect(self.save_to_txt2)
        self.run_button2.clicked.connect(self.run_python_file_mpiexec2)
        
        # 删除文件按钮
        self.delete_button = QPushButton("清除所有")
        self.delete_button.clicked.connect(self.delete_all_result)

        # 退出按钮
        self.exit_button = QPushButton("退出")
        self.exit_button.clicked.connect(self.close)  # 点击时关闭窗口

        # 创建垂直布局
        layout = QVBoxLayout()
        layout.addWidget(label)
        layout.addLayout(form_layout)
        layout.addWidget(self.save_button)  # 添加保存按钮
        layout.addWidget(self.run_button)  # 添加运行按钮
        layout.addWidget(self.run_button2)  # 添加运行按钮
        layout.addWidget(self.delete_button)  # 添加删除按钮
        

        # 添加按钮
        button = QPushButton("切换几何结构")
        button.clicked.connect(self.show_page1)
        layout.addWidget(button)

        layout.addWidget(self.exit_button)  # 添加退出按钮

        self.page2.setLayout(layout)

    def show_page1(self):
        self.stacked_widget.setCurrentIndex(0)

    def show_page2(self):
        self.stacked_widget.setCurrentIndex(1)
        
    def on_checkbox_state_changed(self, state, step_input, end_input, index):
        """复选框状态变化时，决定是否修改 step_input 的值"""
        if state == Qt.Checked:
            # 如果选择单值，将 step_input 设置为 end_input 的值
            step_input.setText(end_input.text())
        else:
            # 取消单值时，恢复默认的 step_input 值（可以根据需求修改）
            step_input.setText(self.default_values[index][2]) 

    def save_to_txt1(self):
        try:
            with open("Geometry/parameters.txt", "w") as file:
                for name, input_widget in self.inputs.items():
                    
                    if name in ["geometry1", "colling_type1","material1","num_cores1"]: 
                        continue
                    if isinstance(input_widget, QLineEdit):
                        # 对于 "geometry", "material" 这些只有一个输入框的项
                        file.write(f"{name}: {input_widget.text()}\n")
                    elif isinstance(input_widget, QComboBox):
                        # 对于 "colling_type" 下拉框，保存当前选中的选项
                        file.write(f"{name}: {input_widget.currentText()}\n")
                    elif isinstance(input_widget, tuple):
                        # 对于 "start", "end", "step" 这样的复合输入框
                        start_val, end_val, step_val = input_widget[0].text(), input_widget[1].text(), input_widget[2].text()
                        file.write(f"{name}: {start_val} {end_val} {step_val}\n")
                    else:
                        # 处理其它复杂的输入类型，比如复选框
                        pass

        except Exception as e:
            print(f"保存文件时发生错误: {e}")
    def save_to_txt2(self):
        try:
            with open("src/parameters.txt", "w") as file:
                for name, input_widget in self.inputs.items():
                    if name in ["geometry", "colling_type","material","num_cores"]: 
                        continue
                    if isinstance(input_widget, QLineEdit):
                        # 对于 "geometry", "material" 这些只有一个输入框的项
                        file.write(f"{name}: {input_widget.text()}\n")
                    elif isinstance(input_widget, QComboBox):
                        # 对于 "colling_type" 下拉框，保存当前选中的选项
                        file.write(f"{name}: {input_widget.currentText()}\n")
                    elif isinstance(input_widget, tuple):
                        # 对于 "start", "end", "step" 这样的复合输入框
                        start_val, end_val, step_val = input_widget[0].text(), input_widget[1].text(), input_widget[2].text()
                        file.write(f"{name}: {start_val} {end_val} {step_val}\n")
                    else:
                        # 处理其它复杂的输入类型，比如复选框
                        pass

        except Exception as e:
            print(f"保存文件时发生错误: {e}")


    def run_python_file1(self):
        """运行特定的Python文件"""
        try:
            subprocess.run(["python", "main.py"], check=True, cwd="Geometry")
        except subprocess.CalledProcessError as e:
            print(f"运行 Python 文件失败: {e}")
        except FileNotFoundError:
            print("找不到 Python 文件，请确保文件路径正确。")

    def run_python_file_mpiexec1(self):
        """运行特定的Python文件"""
        num_cores = self.inputs["num_cores"].text()

        try:
            subprocess.run(["C:\\Program Files\\Microsoft MPI\\Bin\\mpiexec", "-n", num_cores, "python", "main.py"], check=True, cwd="Geometry")
        except subprocess.CalledProcessError as e:
            print(f"运行 Python 文件失败: {e}")
        except FileNotFoundError:
            print("找不到 Python 文件，请确保文件路径正确。")

    def run_python_file2(self):
        """运行特定的Python文件"""
        try:
            subprocess.run(["python", "main.py"], check=True, cwd="src")
        except subprocess.CalledProcessError as e:
            print(f"运行 Python 文件失败: {e}")
        except FileNotFoundError:
            print("找不到 Python 文件，请确保文件路径正确。")

    def run_python_file_mpiexec2(self):
        """运行特定的Python文件"""
        num_cores = self.inputs["num_cores"].text()

        try:
            subprocess.run(["C:\\Program Files\\Microsoft MPI\\Bin\\mpiexec", "-n", num_cores, "python", "main.py"], check=True, cwd="src")
        except subprocess.CalledProcessError as e:
            print(f"运行 Python 文件失败: {e}")
        except FileNotFoundError:
            print("找不到 Python 文件，请确保文件路径正确。")
            
    def delete_all_result(self):
        """删除所有计算结果"""
        # 删除文件及其内容
        directory_path = "out/Deformation/"
        for file in glob.glob(os.path.join(directory_path, "*")):
            if os.path.isfile(file):
                os.remove(file)
        
        # 删除其他文件
        file_name = "out/GeometryParameters.dat"
        if os.path.exists(file_name):
            os.remove(file_name)

        file_name = "out/index.txt"
        if os.path.exists(file_name):
            os.remove(file_name)
            with open(file_name, 'wb') as file:
                file.write(b'0')
        
        

if __name__ == "__main__":
    app = QApplication([])
    window = MainWindow()
    window.show()
    app.exec_()

from IPython import get_ipython
get_ipython().magic('reset -sf')  # 重置内核

In [6]:
from IPython import get_ipython
get_ipython().magic('reset -sf')  # 重置内核


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_31192\1691423192.py:2: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  get_ipython().magic('reset -sf')  # 重置内核


In [6]:
import os
import glob
import sys
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import subprocess
from PIL import Image
import numpy as np
import itertools


In [9]:
subprocess.run(["python", "Geometry/main.py"], check=True)

CalledProcessError: Command '['python', 'Geometry/main.py']' returned non-zero exit status 1.